Proof of <a class="ProveItLink" href="../../../../../../_theory_nbs_/theory.ipynb">proveit</a>.<a class="ProveItLink" href="../../../../../_theory_nbs_/theory.ipynb">physics</a>.<a class="ProveItLink" href="../../../../_theory_nbs_/theory.ipynb">quantum</a>.<a class="ProveItLink" href="../../theory.ipynb">QEC2</a>.<a class="ProveItLink" href="../../theorems.ipynb#mal_set_contains_non_minority_subset_of_bufilo">mal_set_contains_non_minority_subset_of_bufilo</a> theorem
========

In [ ]:
import proveit
theory = proveit.Theory() # the theorem's theory
from proveit import m, n, x, A, B, defaults, display_provers, Lambda # useful imports
from proveit.logic import Equals, Exists, InSet
from proveit.logic.sets import Difference, Disjoint, Intersect, SubsetEq
from proveit.logic.sets.symmetric_difference import sym_diff_as_union
from proveit.numbers import two, LessEq, Natural, Neg
from proveit.physics.quantum.QEC2 import (
    b_star, c_star, mal_set_property, Weight,
    difference_is_subset_of_symmetric_difference,
    weight_of_differences_inequality,
    binary_disjoint_weight_additivity
)


In [ ]:
%proving mal_set_contains_non_minority_subset_of_bufilo

In [ ]:
defaults.assumptions = mal_set_contains_non_minority_subset_of_bufilo.all_conditions()

We pull in and instantiate `mal_set_property` so that we can rewrite $b$ as $m \Delta c$. In `mal_set_property`, think of the posited $c$ as the correction offered by the decoder. The symmetric difference $m \Delta c$ of the malignant set $m$ and the correction $c$ then should give the BUFILO $b$.

In [ ]:
exists_c_exists_b = mal_set_property.instantiate()

In [ ]:
weight_ineq_assumption, exists_b_assumption = exists_c_exists_b.choose(c_star)

In [ ]:
b_star_in_buf_assumption, b_star_as_sym_diff = exists_b_assumption.choose(b_star)

We also pull in and instantiate `sym_diff_as_union` so that we can rewrite $b$ as $(m - c) \cup (c - m)$, and rewrite $w(b)$ as $w(m - c) + w(c - m)$.

In [ ]:
m_delta_c_star_as_union = sym_diff_as_union.instantiate({A:m, B:c_star})

In [ ]:
b_star_as_union = m_delta_c_star_as_union.sub_right_side_into(b_star_as_sym_diff)

In [ ]:
weight_b_star_as_union = b_star_as_union.substitution(Lambda(x, Weight(x)))

In [ ]:
weight_of_union_as_sum = binary_disjoint_weight_additivity.instantiate({A:Difference(m, c_star), B:Difference(c_star, m)})

In [ ]:
weight_b_star_as_sum = weight_of_union_as_sum.sub_right_side_into(weight_b_star_as_union)

Now we begin the main proof. We begin by defining $m' = (m - c^{*})$ and then showing that $m' \subseteq m$ and $m' \subseteq b$.

In [ ]:
m_prime = Difference(m, c_star)

In [ ]:
SubsetEq(m_prime, m).prove()

In [ ]:
m_prime_subset_b = difference_is_subset_of_symmetric_difference.instantiate({A:m, B:c_star})

In [ ]:
b_star_as_sym_diff.sub_right_side_into(m_prime_subset_b)

##### Now we want to show the main property: $w(m') \geq \frac{1}{2}w(b)$.

In [ ]:
weight_inequality = weight_of_differences_inequality.instantiate({A:m, B:c_star})

In [ ]:
weight_inequality = weight_inequality.add(LessEq(Weight(Difference(m, c_star)), Weight(Difference(m, c_star))))

In [ ]:
weight_inequality = weight_inequality.inner_expr().rhs.commute(0, 1)

In [ ]:
weight_inequality = weight_b_star_as_sum.sub_right_side_into(weight_inequality)

In [ ]:
weight_inequality = weight_inequality.divide_both_sides(two)

In [ ]:
conclusion = (mal_set_contains_non_minority_subset_of_bufilo.instance_expr.conclude_via_example((m_prime, b_star)))

#### Eliminate our Skolem constants

In [ ]:
conclusion_elim_b_star = conclusion.eliminate(b_star)

In [ ]:
conclusion_elim_c_star = conclusion_elim_b_star.eliminate(c_star)

In [ ]:
qed_proof = %qed

# Code for printing proof info

In [ ]:
thm = theory.get_theorem("mal_set_contains_non_minority_subset_of_bufilo")

In [ ]:
# STEP 1: Get axioms and conjectures. In the paper, I manually trimmed down this
#         list to include only "interesting" properties.

axioms, conjectures, _ = thm.all_requirements(sort_key=str)

print("Axioms:")
print(r"\begin{itemize}")
for ax in axioms:
    print(r"\item $" + ax.proven_truth.expr.latex() + r"$")
print(r"\end{itemize}")

print("\nUnproven conjectures:")
print(r"\begin{itemize}")
for cj in conjectures:
    print(r"\item $" + cj.proven_truth.expr.latex() + r"$")
print(r"\end{itemize}")

In [ ]:
qed_proof = conclusion_elim_c_star.proof()

In [ ]:
# STEP 2: Get proof steps using the `proof_to_latex_table` function.
#         Something like this function could probably be moved back to Prove-It. 

def proof_to_latex_table(proof):
    esc = lambda s: "".join({
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }.get(c, c) for c in str(s))

    steps = proof.enumerated_proof_steps()
    step_nums = {step: i for i, step in enumerate(steps)}
    any_marked = False

    lines = [
        r"\begin{longtable}{r l l p{0.62\textwidth}}",
        r"\textbf{\#} & \textbf{Step type} & \textbf{Reqs.} & \textbf{Statement} \\",
        r"\hline",
        r"\endhead",
    ]

    for i, step in enumerate(steps):
        reqs = []
        for j, req in enumerate(step.required_proofs):
            marked = j in step.marked_required_truth_indices
            any_marked = any_marked or marked
            reqs.append(str(step_nums[req]) + (r"$^\ast$" if marked else ""))

        lines.append(
            rf"{i} & {esc(step.step_type())} & {', '.join(reqs)} & $\displaystyle {step.proven_truth.latex()}$ \\"
        )

        if step.step_type() == "instantiation" and hasattr(step, "mapping"):
            mapping = ", ".join(
                r"$%s \mapsto %s$" % (k.latex(), step.mapping[k].latex())
                for k in step.mapping_key_order
            )
            if mapping:
                lines.append(
                    r" & \multicolumn{3}{p{0.82\textwidth}}{\emph{Instantiation:} %s} \\"
                    % mapping
                )

        if step.step_type() in ("axiom", "theorem", "conjecture") and hasattr(step, "theory"):
            lines.append(
                r" & \multicolumn{3}{p{0.82\textwidth}}{\texttt{%s}} \\"
                % esc(f"{step.theory}.{step.name}")
            )

    if any_marked:
        lines.append(r"\multicolumn{4}{l}{$^\ast$ equality replacement requirement} \\")

    lines.append(r"\end{longtable}")
    return "\n".join(lines)


In [ ]:
print(proof_to_latex_table(qed_proof))